# 04 — Custom and Cross-Column Business Rules

Phase 3 made raw input manageable.

Phase 4 answers a different question:

> What if every column looks individually valid, but the **business relationship between columns is wrong**?

We will cover:

- `@pa.check`
- `@pa.dataframe_check`
- custom check names and errors
- floating-point-safe total validation
- avoiding cascading failures

## 1. Project setup

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [2]:
import numpy as np
import pandas as pd
import pandera.pandas as pa

from pandera_lab import (
    TOTAL_ABSOLUTE_TOLERANCE,
    expected_order_total,
    load_orders_csv,
    summarize_failure_cases,
    total_matches_formula,
)
from pandera_lab.schemas import Phase4OrderSchema as OrderSchema
from pandera_lab.validation import ValidationResult


In [3]:
def validate_orders(df: pd.DataFrame) -> ValidationResult:
    """Frozen Phase-4 validator used by this historical notebook."""
    try:
        validated = OrderSchema.validate(df, lazy=True)
    except pa.errors.SchemaErrors as exc:
        return ValidationResult(
            is_valid=False,
            data=None,
            failure_cases=exc.failure_cases.copy(),
        )

    return ValidationResult(
        is_valid=True,
        data=validated,
        failure_cases=pd.DataFrame(),
    )


## 2. Inspect the new checks

In [4]:
OrderSchema.to_schema()

<Schema DataFrameSchema(columns={'order_id': <Schema Column(name=order_id, type=DataType(int64))>, 'customer_id': <Schema Column(name=customer_id, type=DataType(str))>, 'product_id': <Schema Column(name=product_id, type=DataType(str))>, 'quantity': <Schema Column(name=quantity, type=DataType(int64))>, 'unit_price': <Schema Column(name=unit_price, type=DataType(float64))>, 'discount': <Schema Column(name=discount, type=DataType(float64))>, 'total': <Schema Column(name=total, type=DataType(float64))>, 'status': <Schema Column(name=status, type=DataType(str))>, 'order_date': <Schema Column(name=order_date, type=DataType(datetime64[ns]))>}, checks=[<Check total_matches_formula: total must equal unit_price * quantity * (1 - discount) within the configured currency tolerance>], parsers=[], index=None, dtype=None, coerce=False, strict=filter, name=Phase4OrderSchema, ordered=False, unique=None, report_duplicates=all, unique_column_names=False, add_missing_columns=False, title=None, description

Look for:

```text
customer_id_not_blank
product_id_not_blank
total_matches_formula
```

The first two are column-level custom checks.

The third is a DataFrame-level check.

## 3. Column custom check: non-blank IDs

In [5]:
blank_customer = pd.DataFrame({
    "order_id": ["5001"],
    "customer_id": ["   "],
    "product_id": ["P501"],
    "quantity": ["1"],
    "unit_price": ["100.0"],
    "discount": ["0.0"],
    "total": ["100.0"],
    "status": ["paid"],
    "order_date": ["2026-08-25"],
})

blank_result = validate_orders(blank_customer)
blank_result.failure_cases

,schema_context,column,check,check_number,failure_case,index
0,Column,customer_id,customer_id must contain non-whitespace charac...,0,,0


`nullable=False` catches missing values.

But `"   "` is present data, so a custom semantic rule is useful.

## 4. Why cross-column validation?

In [6]:
reference = pd.read_csv(
    ROOT / "data" / "reference" / "orders_valid.csv"
)

reference[[
    "unit_price",
    "quantity",
    "discount",
    "total",
]]

,unit_price,quantity,discount,total
0,100.0,2,0.10,180.0
1,50.0,1,0.00,50.0
2,20.0,3,0.25,45.0
3,250.0,1,0.20,200.0
4,15.0,4,0.00,60.0


In [7]:
comparison = reference[[
    "order_id",
    "unit_price",
    "quantity",
    "discount",
    "total",
]].copy()

comparison["expected_total"] = expected_order_total(reference)
comparison

,order_id,unit_price,quantity,discount,total,expected_total
0,2001,100.0,2,0.10,180.0,180.0
1,2002,50.0,1,0.00,50.0,50.0
2,2003,20.0,3,0.25,45.0,45.0
3,2004,250.0,1,0.20,200.0,200.0
4,2005,15.0,4,0.00,60.0,60.0


Every row should satisfy:

```text
total = unit_price * quantity * (1 - discount)
```

## 5. Break the total

In [8]:
wrong_total = reference.copy()
wrong_total.loc[0, "total"] = 999.0

wrong_result = validate_orders(wrong_total)

print("is_valid:", wrong_result.is_valid)
wrong_result.failure_cases

is_valid: False


,schema_context,column,check,check_number,failure_case,index
0,DataFrameSchema,order_id,total must equal unit_price * quantity * (1 - ...,0,2001,0
1,DataFrameSchema,customer_id,total must equal unit_price * quantity * (1 - ...,0,C101,0
2,DataFrameSchema,product_id,total must equal unit_price * quantity * (1 - ...,0,P101,0
3,DataFrameSchema,quantity,total must equal unit_price * quantity * (1 - ...,0,2,0
4,DataFrameSchema,unit_price,total must equal unit_price * quantity * (1 - ...,0,100.0,0
5,DataFrameSchema,discount,total must equal unit_price * quantity * (1 - ...,0,0.1,0
6,DataFrameSchema,total,total must equal unit_price * quantity * (1 - ...,0,999.0,0
7,DataFrameSchema,status,total must equal unit_price * quantity * (1 - ...,0,paid,0
8,DataFrameSchema,order_date,total must equal unit_price * quantity * (1 - ...,0,2026-08-01 00:00:00,0


Phase 3 would have accepted this value because `999.0` is a valid float.

Phase 4 rejects it because the **relationship** is wrong.

## 6. The floating-point equality trap

In [9]:
x = 0.1 * 3
x, 0.3, x == 0.3

(0.30000000000000004, 0.3, False)

Exact equality is not always a robust representation of business equality for binary floats.

In [10]:
np.isclose(
    0.1 * 3,
    0.3,
    rtol=0.0,
    atol=TOTAL_ABSOLUTE_TOLERANCE,
)

np.True_

## 7. Why half a cent?

In [11]:
TOTAL_ABSOLUTE_TOLERANCE

0.005

The project uses `0.005`:

- floating representation noise passes,
- a full one-cent discrepancy does not.

This is a **business-shaped tolerance**, not a random epsilon.

## 8. Representation noise should pass

In [12]:
float_noise = pd.DataFrame({
    "unit_price": [0.1],
    "quantity": [3],
    "discount": [0.0],
    "total": [0.30000000000000004],
})

total_matches_formula(float_noise)

0    True
dtype: bool

## 9. One cent should fail

In [13]:
one_cent_wrong = pd.DataFrame({
    "unit_price": [100.0],
    "quantity": [1],
    "discount": [0.0],
    "total": [100.01],
})

total_matches_formula(one_cent_wrong)

0    False
dtype: bool

## 10. Avoid cascading errors

In [14]:
unparseable = pd.DataFrame({
    "unit_price": ["50.0"],
    "quantity": ["two"],
    "discount": ["0.0"],
    "total": ["100.0"],
})

total_matches_formula(unparseable)

0    True
dtype: bool

Why is the result `True` here?

Not because the order is valid.

It means:

> This dataframe-level rule deliberately declines to add a second error when its prerequisite data is unparseable.

The column-level quantity coercion rule remains responsible for invalidating the row.

## 11. Confirm it through the full schema

In [15]:
bad_quantity = pd.DataFrame({
    "order_id": ["5101"],
    "customer_id": ["C510"],
    "product_id": ["P510"],
    "quantity": ["two"],
    "unit_price": ["50.0"],
    "discount": ["0.0"],
    "total": ["100.0"],
    "status": ["paid"],
    "order_date": ["2026-08-25"],
})

quantity_result = validate_orders(bad_quantity)
quantity_result.failure_cases

,schema_context,column,check,check_number,failure_case,index
0,Column,quantity,coerce_dtype('int64'),None,two,0
1,Column,quantity,dtype('int64'),None,two,0
2,Column,quantity,greater_than(0),0,"TypeError(""'>' not supported between instances...",None


You should see the quantity/coercion problem without a misleading `total_matches_formula` failure.

## 12. Revisit the deliberately messy raw dataset

In [16]:
raw_df = load_orders_csv(
    ROOT / "data" / "raw" / "orders.csv"
)

raw_result = validate_orders(raw_df)
summary = summarize_failure_cases(raw_result.failure_cases)
summary

,column,check,failures
0,customer_id,not_nullable,1
1,customer_id,total must equal unit_price * quantity * (1 - ...,1
2,discount,greater_than_or_equal_to(0),1
3,discount,less_than_or_equal_to(1),1
4,discount,total must equal unit_price * quantity * (1 - ...,1
5,order_date,coerce_dtype('datetime64[ns]'),1
6,order_date,dtype('datetime64[ns]'),1
7,order_date,total must equal unit_price * quantity * (1 - ...,1
8,order_id,field_uniqueness,2
9,order_id,total must equal unit_price * quantity * (1 - ...,1


Search the summary for:

```text
total_matches_formula
```

The row with order `1006` is finally detectable as a business-rule violation.

In [17]:
summary[
    summary["check"].astype(str).str.contains(
        "total_matches_formula",
        regex=False,
    )
]

,column,check,failures


## 13. Inspect the offending raw row

In [18]:
raw_df.loc[raw_df["order_id"].astype(str) == "1006"]

,order_id,customer_id,product_id,quantity,unit_price,discount,total,status,order_date,internal_note
5,1006,C006,P006,2,75.0,0.1,100.0,shipped,2026-08-06,wrong total


In [19]:
row_1006 = raw_df.loc[
    raw_df["order_id"].astype(str) == "1006"
].copy()

row_1006["expected_total"] = expected_order_total(row_1006)
row_1006[[
    "order_id",
    "unit_price",
    "quantity",
    "discount",
    "total",
    "expected_total",
]]

,order_id,unit_price,quantity,discount,total,expected_total
5,1006,75.0,2,0.1,100.0,135.0


# Phase-4 checkpoint

You should now be able to explain:

1. why built-in checks are preferred when possible,
2. when to use `@pa.check`,
3. when to use `@pa.dataframe_check`,
4. why `total` needs the whole DataFrame,
5. why float `==` is risky,
6. how a business-shaped tolerance is chosen,
7. what a cascading validation failure is,
8. why helper functions make business rules easier to test.

## Next phase

Phase 5 moves from a validation library to an actual typed pipeline:

```text
RawSchema
   ↓
@pa.check_types
   ↓
transform
   ↓
OutputSchema
```